# Implementazione dell'algoritmo Apriori

In [1]:
try:
  if dataset:
    print("Already loaded")
except:
  !pip install ucimlrepo polars
  from ucimlrepo import fetch_ucirepo
  dataset = fetch_ucirepo(id=360)

dataset.data.features

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9352,4/4/2005,10:00:00,3.1,1314,-200,13.5,1101,472,539,190,1374,1729,21.9,29.3,0.7568
9353,4/4/2005,11:00:00,2.4,1163,-200,11.4,1027,353,604,179,1264,1269,24.3,23.7,0.7119
9354,4/4/2005,12:00:00,2.4,1142,-200,12.4,1063,293,603,175,1241,1092,26.9,18.3,0.6406
9355,4/4/2005,13:00:00,2.1,1003,-200,9.5,961,235,702,156,1041,770,28.3,13.5,0.5139


## Import del dataset

In [2]:
import polars as pl

aq = pl.from_pandas(dataset.data.features)

aq = (aq
    # Drop specified columns
    .drop(["Date", "Time", "RH", "AH"])

    # Drop GT columns
    .drop(filter(lambda x: x[-4:] == '(GT)', aq.columns))

    # Rename columns
    .rename({
        'PT08.S1(CO)': 'CO',
        'PT08.S2(NMHC)': 'NMHC',
        'PT08.S3(NOx)': 'NOX',
        'PT08.S4(NO2)': 'NOO',
        'PT08.S5(O3)': 'OOO'
    })

    # Filter out rows with invalid values
    .filter(pl.any_horizontal(pl.all().ne(-200)))
)

aq

CO,NMHC,NOX,NOO,OOO,T
i64,i64,i64,i64,i64,f64
1360,1046,1056,1692,1268,13.6
1292,955,1174,1559,972,13.3
1402,939,1140,1555,1074,11.9
1376,948,1092,1584,1203,11.0
1272,836,1205,1490,1110,11.2
…,…,…,…,…,…
1314,1101,539,1374,1729,21.9
1163,1027,604,1264,1269,24.3
1142,1063,603,1241,1092,26.9


## Encoding degli intervalli del dataset

In [3]:
# Create encoders dictionary first
encoder = {
    col: sorted(aq.get_column(col).unique().to_list())
    for col in aq.columns
}

# Create the encoded DataFrame using polars expressions
df = pl.DataFrame()

# Encode each column using match patterns
df = aq.select([
    pl.col(column).replace_strict(
        dict(zip(values, range(len(values)))),
    ).alias(column)
    for column, values in encoder.items()
])

df

CO,NMHC,NOX,NOO,OOO,T
i64,i64,i64,i64,i64,i64
665,622,699,985,976,145
597,531,817,852,680,142
707,515,783,848,782,128
681,524,735,877,911,119
577,412,847,783,818,121
…,…,…,…,…,…
619,677,182,667,1420,228
468,603,247,557,977,252
447,639,246,534,800,278


## Applicazione di Apriori

In [4]:
from frozendict import frozendict

def get_itemset_freq(itemset):
  # Construct query string for all ranges
  conditions = [
        (pl.col(col).is_between(range_vals[0], range_vals[1]))
        for col, range_vals in itemset.items()
  ]

  # Combine all conditions
  combined_filter = pl.reduce(lambda x, y: x & y, conditions)

  # Apply filter and count matching rows
  satisfied_rows = len(df.filter(combined_filter))

  return satisfied_rows

def get_support(itemset):
  return get_itemset_freq(itemset) / len(df)

def apriori(min_support, min_step):
  # Initialize itemset
  SWk = { frozendict({k:(0, len(v) - 1) for k,v in encoder.items()}) }

  res = pl.DataFrame()
  shirinking = 0

  while bool(SWk):
    shirinking += 1
    print(f"Calcolo gli itemset con {shirinking} shirinkings, al passo precedente ho trovato {len(SWk)} itemsets")

    # Generate smaller frequent itemsets
    Wk = set()
    for itemset in SWk:
      for col in itemset:
        if itemset[col][1] - itemset[col][0] > min_step[col]:
          Wk.add(itemset.set(col, (itemset[col][0] + min_step[col], itemset[col][1])))
          Wk.add(itemset.set(col, (itemset[col][0], itemset[col][1] - min_step[col])))

    # Filter not supported frequent itemsets
    SWk_new = set()
    R = []
    for itemset in Wk:
      itemset_support = get_support(itemset)

      if itemset_support >= min_support:
        SWk_new.add(itemset)
        R.append({'support': itemset_support} | {k: v for k, v in itemset.items() if v[0] != 0 or v[1] != len(encoder[k]) - 1 })

    SWk = SWk_new
    res = pl.concat([pl.DataFrame(R), res], how="diagonal")

  return res

frequent_itemsets = apriori(0.1, {k: (len(v) // 4) for k,v in encoder.items()})

Calcolo gli itemset con 1 shirinkings, al passo precedente ho trovato 1 itemsets
Calcolo gli itemset con 2 shirinkings, al passo precedente ho trovato 12 itemsets
Calcolo gli itemset con 3 shirinkings, al passo precedente ho trovato 78 itemsets
Calcolo gli itemset con 4 shirinkings, al passo precedente ho trovato 354 itemsets
Calcolo gli itemset con 5 shirinkings, al passo precedente ho trovato 1203 itemsets
Calcolo gli itemset con 6 shirinkings, al passo precedente ho trovato 3174 itemsets
Calcolo gli itemset con 7 shirinkings, al passo precedente ho trovato 6635 itemsets
Calcolo gli itemset con 8 shirinkings, al passo precedente ho trovato 11137 itemsets
Calcolo gli itemset con 9 shirinkings, al passo precedente ho trovato 15107 itemsets
Calcolo gli itemset con 10 shirinkings, al passo precedente ho trovato 16524 itemsets
Calcolo gli itemset con 11 shirinkings, al passo precedente ho trovato 14412 itemsets
Calcolo gli itemset con 12 shirinkings, al passo precedente ho trovato 9826 it

In [6]:
(
    frequent_itemsets
    .with_columns([
        pl.col(col).list.eval(
            pl.element().replace_strict(range(len(index)), index)
            ).alias(col)
            for col, index in encoder.items()])
    .sort(by="support")
)

support,CO,NMHC,NOX,NOO,OOO,T
f64,list[i64],list[i64],list[i64],list[i64],list[i64],list[f64]
0.1001,"[955, 1215]","[383, 1357]","[662, 967]","[1507, 1911]","[727, 1164]","[10.0, 31.7]"
0.1001,"[955, 1215]",null,"[662, 967]","[1507, 1911]","[727, 1164]","[10.0, 31.7]"
0.1001,"[955, 1215]","[735, 1046]","[322, 1276]","[1106, 1509]","[727, 1164]","[-1.9, 31.7]"
0.1001,"[647, 1215]","[735, 1046]","[662, 1276]","[1106, 1509]","[727, 1164]","[10.0, 44.6]"
0.1001,"[955, 2040]","[735, 1357]","[662, 967]","[1507, 2775]","[727, 1607]","[20.9, 31.7]"
…,…,…,…,…,…,…
0.907908,null,null,null,"[551, 1911]",null,null
0.918696,null,null,null,null,null,"[-1.9, 31.7]"
0.927705,null,"[383, 1357]",null,null,null,null


## Estrazione delle regole di associazione

In [7]:
from itertools import combinations
import math

def get_p_value(X, Y):
  p_value = 1
  N = len(df)
  Nxy = get_itemset_freq(X | Y)
  Cx = get_itemset_freq(X)
  Cy = get_itemset_freq(Y)
  bin_X_Cx = math.comb(N, Cx)

  for k in range(0, Nxy):
    p_value -= math.comb(Cy, k) * math.comb(N - Cy, Cx - k) / bin_X_Cx

  return p_value

def mine_rules(R, min_confidence, min_lift = 1):
  res = pl.DataFrame()

  for itemset in R.rows(named=True):
    itemset = { k: v for k,v in itemset.items() if v }

    rules = []
    support = itemset.pop('support')

    if len(itemset) < 2:
      continue

    for i in range(1, len(itemset)):
      for combination in combinations(itemset.items(), i):
        X = frozendict(combination)
        Y = frozendict({k:v for k,v in itemset.items() if k not in X})

        confidence = support / get_support(X)
        if confidence < min_confidence:
          continue

        lift = confidence / get_support(Y)
        if lift <= min_lift:
          continue

        p_value = get_p_value(X, Y)

        rule = {"X": list(X.keys()), "Y": list(Y.keys()), "support": support, "confidence": confidence, "lift": lift, "p_value": p_value} | itemset
        rules.append(rule)

    res = pl.concat([pl.DataFrame(rules), res], how="diagonal")

  return res

association_rules = mine_rules(frequent_itemsets, 0.99, 1.54)

In [12]:
association_rules

X,Y,support,confidence,lift,p_value,CO,NMHC,NOO,OOO,NOX,T
list[str],list[str],f64,f64,f64,f64,list[i64],list[i64],list[i64],list[i64],list[i64],list[i64]
"[""NMHC""]","[""CO"", ""NOO"", ""OOO""]",0.322322,0.994509,1.642777,4.2880e-16,"[260, 1040]","[622, 1244]","[400, 1602]","[435, 1742]",null,null
"[""NMHC""]","[""CO"", ""NOX"", ""NOO""]",0.3221,0.993823,1.541926,5.1952e-16,"[260, 1040]","[622, 1244]","[400, 1602]",null,"[0, 915]",null
"[""CO""]","[""NMHC"", ""NOO"", ""OOO""]",0.267935,0.997516,1.589343,-3.5556e-16,"[520, 1040]","[311, 1244]","[400, 1602]","[435, 1742]",null,null
"[""NOX""]","[""CO"", ""NMHC"", ""NOO""]",0.254477,0.993487,1.596789,5.3562e-16,"[260, 1040]","[311, 1244]","[400, 1602]",null,"[0, 305]",null
"[""CO""]","[""NMHC"", ""NOX""]",0.285508,0.999221,1.572004,3.9960e-16,"[0, 260]","[0, 622]",null,null,"[305, 1220]",null
…,…,…,…,…,…,…,…,…,…,…,…
"[""CO"", ""NOX"", … ""T""]","[""NMHC"", ""OOO""]",0.105439,0.995798,1.619906,-4.3958e-16,"[0, 520]","[0, 622]","[400, 802]","[0, 872]","[610, 915]","[109, 326]"
"[""NMHC"", ""NOX"", … ""T""]","[""CO"", ""OOO""]",0.105439,0.997895,1.541328,3.2573e-16,"[0, 520]","[0, 622]","[400, 802]","[0, 872]","[610, 915]","[109, 326]"
"[""NOX"", ""NOO"", … ""T""]","[""CO"", ""NMHC""]",0.105439,0.995798,1.55061,-4.2837e-17,"[0, 520]","[0, 622]","[400, 802]","[0, 872]","[610, 915]","[109, 326]"


In [13]:
# Decode columns with encoder dictionary
association_rules_mapped = (
    association_rules
    .with_columns([
        pl.col(col).list.eval(
            pl.element().replace_strict(range(len(index)), index)
            ).alias(col)
            for col, index in encoder.items()])
    .sort(by="support")
)

In [14]:
association_rules_mapped.sort(by='lift')

X,Y,support,confidence,lift,p_value,CO,NMHC,NOO,OOO,NOX,T
list[str],list[str],f64,f64,f64,f64,list[i64],list[i64],list[i64],list[i64],list[i64],list[f64]
"[""NMHC"", ""T""]","[""CO"", ""NOX"", ""NOO""]",0.223668,0.992596,1.540023,1.4548e-16,"[955, 2040]","[1046, 1357]","[1106, 2775]",null,"[322, 1276]","[10.0, 44.6]"
"[""NMHC"", ""OOO"", ""T""]","[""CO"", ""NOX"", ""NOO""]",0.223668,0.992596,1.540023,1.4548e-16,"[955, 2040]","[1046, 1357]","[1106, 2775]","[727, 2523]","[322, 1276]","[10.0, 44.6]"
"[""NMHC"", ""T""]","[""CO"", ""NOX"", ""NOO""]",0.194083,0.992605,1.540037,2.4309e-16,"[955, 2040]","[1046, 1357]","[1106, 2775]",null,"[322, 1276]","[10.0, 31.7]"
"[""NMHC"", ""OOO"", ""T""]","[""CO"", ""NOX"", ""NOO""]",0.194083,0.992605,1.540037,2.4309e-16,"[955, 2040]","[1046, 1357]","[1106, 2775]","[727, 2523]","[322, 1276]","[10.0, 31.7]"
"[""NMHC"", ""OOO"", ""T""]","[""CO"", ""NOX"", ""NOO""]",0.164498,0.992617,1.540056,4.2973e-16,"[955, 2040]","[1046, 1357]","[1106, 2775]","[1162, 2523]","[322, 1276]","[10.0, 44.6]"
…,…,…,…,…,…,…,…,…,…,…,…
"[""CO"", ""NOX"", … ""T""]","[""NMHC""]",0.128017,0.995675,3.072104,-4.6519e-16,"[1215, 2040]","[1046, 2214]","[1507, 2775]","[1162, 2523]","[322, 662]","[10.0, 31.7]"
"[""CO"", ""NOX"", … ""T""]","[""NMHC""]",0.135802,0.995922,3.072866,-1.4057e-16,"[1215, 2040]","[1046, 2214]","[1507, 2775]","[1162, 2523]","[322, 662]","[10.0, 44.6]"
"[""CO"", ""NOX"", … ""T""]","[""NMHC""]",0.138249,0.995994,3.073088,4.1636e-16,"[1215, 2040]","[1046, 2214]","[1507, 2775]","[1162, 2523]","[322, 662]","[-1.9, 31.7]"


In [15]:
association_rules_mapped.sort(by='p_value')

X,Y,support,confidence,lift,p_value,CO,NMHC,NOO,OOO,NOX,T
list[str],list[str],f64,f64,f64,f64,list[i64],list[i64],list[i64],list[i64],list[i64],list[f64]
"[""NOX"", ""NOO"", … ""T""]","[""CO"", ""NMHC""]",0.165054,1.0,1.557153,-1.1179e-15,"[647, 1215]","[383, 1046]","[551, 1509]","[221, 729]","[662, 2683]","[10.0, 44.6]"
"[""NMHC""]","[""CO"", ""NOO"", ""OOO""]",0.250028,0.99469,1.542739,-9.8641e-16,"[647, 1215]","[383, 735]","[551, 1911]","[221, 1164]",null,null
"[""NMHC"", ""NOX""]","[""CO"", ""NOO"", ""OOO""]",0.250028,0.99469,1.542739,-9.8641e-16,"[647, 1215]","[383, 735]","[551, 1911]","[221, 1164]","[662, 2683]",null
"[""NMHC"", ""OOO""]","[""CO"", ""NOX"", ""NOO""]",0.262485,0.994103,1.54236,-9.5283e-16,"[955, 2040]","[1046, 2214]","[1106, 2775]","[1162, 2523]","[322, 1276]",null
"[""CO"", ""NOX"", ""T""]","[""NMHC"", ""NOO"", ""OOO""]",0.185296,0.991667,1.613769,-9.4886e-16,"[647, 1475]","[383, 1046]","[551, 1911]","[221, 1164]","[967, 2683]","[-1.9, 20.8]"
…,…,…,…,…,…,…,…,…,…,…,…
"[""CO"", ""NOX"", … ""OOO""]","[""NMHC"", ""T""]",0.131465,0.997468,1.556715,1.2140e-15,"[955, 1215]","[735, 2214]","[1507, 2775]","[727, 1164]","[662, 967]","[10.0, 44.6]"
"[""NOX"", ""NOO""]","[""CO"", ""NMHC"", ""OOO""]",0.227561,0.994169,1.549683,1.2628e-15,"[647, 1215]","[383, 1046]","[551, 1509]","[221, 1607]","[967, 2683]",null
"[""OOO""]","[""CO"", ""NMHC"", ""NOX""]",0.248582,0.999106,1.572923,1.2863e-15,"[647, 1475]","[383, 1046]",null,"[221, 729]","[662, 2683]",null
